In [ ]:
# ==============================================================================
# COLAB / ANTIGRAVITY TRAINING — LAUNCHER
#
# This notebook contains NO training code. It runs the script from the
# repository: scripts/colab/colab_clean_cell.py
#
# Why: this notebook used to hold a pasted copy, and the copy went stale --
# on 2026-08-02 it still used train_test_split(random_state=42), a random
# split that lets a validation window share 19 of its 20 days with a
# training window, months after the script had been fixed to split
# chronologically with a purge gap. One copy, in git, reviewed and tested.
#
# WHERE THE PROJECT IS: found, not assumed. Earlier versions of this cell
# required the repository on Google Drive and stopped otherwise; that is one
# valid setup among several. An IDE-hosted notebook usually has the checkout
# as the working directory and only the batch data on Drive.
# ==============================================================================

import os, sys
from pathlib import Path

# Set this if neither guess below finds your checkout.
PROJECT_PATH = None

def _looks_like_project(p):
    p = Path(p)
    return (p / "scripts" / "colab" / "colab_clean_cell.py").exists()

def _find_project():
    if PROJECT_PATH and _looks_like_project(PROJECT_PATH):
        return Path(PROJECT_PATH)
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if _looks_like_project(candidate):
            return candidate
    drive = Path("/content/drive/MyDrive/trading_project")
    if _looks_like_project(drive):
        return drive
    try:
        from google.colab import drive as _d
        _d.mount("/content/drive", force_remount=False)
        if _looks_like_project(drive):
            return drive
    except Exception:
        pass
    return None

project = _find_project()
if project is None:
    raise SystemExit(
        "Could not find the repository. Looked in the working directory and "
        "its parents, then /content/drive/MyDrive/trading_project. Set "
        "PROJECT_PATH at the top of this cell to your checkout."
    )

os.chdir(project)
for path in (str(project), str(project / "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

# One log holding everything the run says. The trainer reports progress with
# print(), which otherwise goes only to this cell's output and is lost the
# moment it scrolls or the session ends -- and a two-hour run that ends in
# "0 predictions" is exactly the one whose log is wanted afterwards.
sys.path.insert(0, str(project / "scripts" / "colab"))
from run_logging import start_run_log
LOG_PATH = start_run_log(name="colab_train")

import subprocess, datetime
print("project:", project)
print("script mtime:", datetime.datetime.fromtimestamp(
    (project / "scripts" / "colab" / "colab_clean_cell.py").stat().st_mtime))
try:
    head = subprocess.run(
        ["git", "-C", str(project), "log", "-1", "--format=%h %ad %s", "--date=short"],
        capture_output=True, text=True, timeout=30)
    print("repo HEAD:", (head.stdout or head.stderr).strip() or "(not a git checkout)")
except Exception as exc:
    print("repo HEAD: unavailable --", exc)

# ------------------------------------------------------------------
# Is this copy of src/ the one that built the batch?
#
# The repository reaches the training machine as a hand-copied mirror --
# the cell, src/, scripts/ -- with no git history to interrogate. That is
# precisely how a copy goes stale without anyone noticing: this notebook
# itself once ran six-week-old training code, complete with a random
# train/test split, while the fix sat in the repo.
#
# prepare records a fingerprint of the code that produced the batch, in
# raw_db_fingerprint.json. Recomputing it here answers the only question
# that matters before a two-hour run: am I about to train with the same
# code that made this data?
# ------------------------------------------------------------------
def _check_code_matches_batch(project, batch_dir):
    fp_file = Path(batch_dir) / "raw_db_fingerprint.json"
    if not fp_file.exists():
        print("ℹ️  No raw_db_fingerprint.json in the batch — cannot verify the code matches.")
        return
    try:
        recorded = json.loads(fp_file.read_text(encoding="utf-8")).get("code_fingerprint")
    except Exception as exc:
        print(f"ℹ️  Could not read the batch fingerprint ({exc}).")
        return
    if not recorded:
        print("ℹ️  The batch records no code fingerprint.")
        return
    try:
        from src.cli.pipeline_executor import PipelineExecutor
        current = PipelineExecutor._compute_code_fingerprint()
    except Exception as exc:
        print(f"ℹ️  Could not recompute the fingerprint here ({exc}).")
        return
    if current == recorded:
        print(f"✅ src/ matches the code that built this batch ({current[:12]}).")
    else:
        # A bare "the hashes differ" would be red after every commit, and a
        # warning that is always on stops being read. The two cases it
        # conflates need different actions, and the timestamps separate them:
        # src OLDER than the batch means this copy is stale, which is the
        # real risk in a hand-copied mirror. src NEWER means the repository
        # simply moved on after the batch was built, which is normal.
        import datetime as _dt
        try:
            built_at = _dt.datetime.fromisoformat(
                json.loads(fp_file.read_text(encoding="utf-8"))["generated_at"]
            )
            newest_src = max(
                p.stat().st_mtime
                for p in (Path(project) / "src").rglob("*.py")
            )
            src_at = _dt.datetime.fromtimestamp(newest_src)
        except Exception:
            built_at = src_at = None

        print(f"⚠️  src/ differs from the code that built this batch.")
        print(f"    batch built by {recorded[:12]}, this copy is {current[:12]}.")
        if built_at and src_at and src_at < built_at:
            print(f"    THIS COPY IS OLDER (src {src_at:%Y-%m-%d %H:%M} < batch {built_at:%Y-%m-%d %H:%M}).")
            print("    Re-copy src/ and scripts/ before training -- you are about")
            print("    to run code the batch has already moved past.")
        elif built_at and src_at:
            print(f"    This copy is NEWER (src {src_at:%Y-%m-%d %H:%M} > batch {built_at:%Y-%m-%d %H:%M}).")
            print("    The repository moved on after the batch was built. Fine if")
            print("    the changes since do not affect feature VALUES; re-run")
            print("    --mode prepare if they do.")

import json

def _find_batch(project):
    """A batch is a directory holding features.parquet and targets.parquet.

    Identified by content rather than by a path anyone has to type. An
    override that points at nothing is ignored with a warning instead of
    being displayed as if it were real -- a run has already died on a
    placeholder from a copied instruction pasted verbatim:
    "/content/drive/MyDrive/.../main_database", ellipsis and all.
    """
    def holds(p):
        p = Path(p)
        return (p / "features.parquet").exists() and (p / "targets.parquet").exists()

    override = os.environ.get("COLAB_BATCH_DIR")
    if override:
        if holds(override):
            return Path(override)
        print(f"⚠️  COLAB_BATCH_DIR points at {override}, which holds no batch — ignoring it.")

    default = project / "data" / "colab" / "accumulated" / "main_database"
    if holds(default):
        return default

    root = project / "data" / "colab" / "accumulated"
    if root.exists():
        found = sorted((d for d in root.iterdir() if d.is_dir() and holds(d)),
                       key=lambda d: (d / "features.parquet").stat().st_mtime,
                       reverse=True)
        if found:
            print(f"📦 found batch: {found[0]}")
            return found[0]

    print(f"❌ No batch found under {root}. A batch holds features.parquet and targets.parquet.")
    return default

def _check_trainer_freshness(project, batch_dir):
    """Is the script about to run from the same copy as everything else?

    _check_code_matches_batch answers this for src/, by fingerprint. It
    cannot answer it for scripts/colab/colab_clean_cell.py: that file is
    deliberately outside the fingerprint, because it does not affect what
    stages 0-3 compute -- it is the trainer, not the feature builder.

    Which left the one file this cell actually executes with no staleness
    signal at all. Twice in a row a run died on a batch path that the
    repository had already been fixed to reject, because the copy on Drive
    predated the fix. Both times the traceback described a missing file, and
    both times the real state was an old script.

    mtime against the batch's build time separates the two cases the way
    the src/ check does: a trainer older than the batch was not re-copied.
    """
    fp_file = Path(batch_dir) / "raw_db_fingerprint.json"
    script = Path(project) / "scripts" / "colab" / "colab_clean_cell.py"
    if not fp_file.exists() or not script.exists():
        return
    try:
        import datetime as _dt
        built_at = _dt.datetime.fromisoformat(
            json.loads(fp_file.read_text(encoding="utf-8"))["generated_at"]
        )
        script_at = _dt.datetime.fromtimestamp(script.stat().st_mtime)
    except Exception:
        return
    if script_at < built_at:
        print(f"⚠️  The trainer script predates this batch "
              f"(script {script_at:%Y-%m-%d %H:%M} < batch {built_at:%Y-%m-%d %H:%M}).")
        print("    scripts/colab/ was probably not re-copied. Copy it, then")
        print("    RESTART THE RUNTIME -- a module already imported keeps its")
        print("    old code, and a traceback whose line numbers do not match")
        print("    the file you are reading is exactly that.")
    else:
        print(f"✅ trainer script is newer than the batch ({script_at:%Y-%m-%d %H:%M}).")


BATCH_DIR = _find_batch(project)
print("batch:", BATCH_DIR)
_check_code_matches_batch(project, BATCH_DIR)
_check_trainer_freshness(project, BATCH_DIR)


In [ ]:
%run "scripts/colab/colab_clean_cell.py"


In [ ]:
# Close the log and show where it is, plus what the run produced.
from run_logging import stop_run_log
stop_run_log()
print("LOG:", LOG_PATH.resolve())

import os
batch_dir = "data/colab/accumulated/main_database"
if os.path.isdir(batch_dir):
    for f in sorted(os.listdir(batch_dir)):
        if not f.startswith('.'):
            size_mb = os.path.getsize(os.path.join(batch_dir, f)) / 1024 / 1024
            print(f"{size_mb:>8.1f} MB  {f}")
else:
    print(f"(no batch directory at {batch_dir})")
